# Capability 06 — Recommendations and uncertainty

Executed evidence over in-memory seed facts. Ranking uses the real catalogue only. No plan is invented.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from telco_digital.application.clock import FixedClock
from telco_digital.application.seed import seed_demo_customers
from telco_digital.infrastructure.memory import InMemoryUnitOfWork
from telco_digital.intelligence.event_memory import EventMemoryService
from telco_digital.intelligence.event_memory.uow import UnitOfWorkEventMemoryQueries
from telco_digital.intelligence.recommendations import (
    DecisionMode,
    PlanRepositoryCatalogue,
    RecommendationService,
)

ROOT = Path(".")
for folder in ("outputs/tables", "outputs/plots", "artifacts"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
AS_OF = datetime.fromisoformat("2026-08-20T12:00:00+00:00")


In [2]:
uow = InMemoryUnitOfWork()
await seed_demo_customers(uow, clock=FixedClock(AS_OF))
service = RecommendationService(
    EventMemoryService(UnitOfWorkEventMemoryQueries(uow)),
    PlanRepositoryCatalogue(uow.plans),
)
document = await service.recommend("U001", AS_OF, destination="SG")
assert document.mode == DecisionMode.SCENARIO_BASED
assert document.primary.plan_code == "ROAM_15"
ranked = pd.DataFrame(
    [
        {
            "plan_code": item.plan_code,
            "score": item.score,
            "confidence": item.confidence,
            "scenario_label": item.scenario_label,
            "reasons": " | ".join(item.reasons),
        }
        for item in document.ranked
    ]
)
ranked


,plan_code,score,confidence,scenario_label,reasons
0,ROAM_15,1.0,0.82,4–7 days,Present in the active catalogue | Same plan as...
1,ROAM_30,0.3,0.60,8–14 days,Present in the active catalogue | Catalogue da...
2,ROAM_5,0.0,0.45,1–3 days,Present in the active catalogue | Historical u...


In [3]:
known = await service.recommend("U001", datetime.fromisoformat("2026-03-16T18:00:00+00:00"), destination="SG")
ask = await service.recommend("U003", AS_OF)
none = await service.recommend("U001", AS_OF, destination="US")
modes = pd.DataFrame(
    [
        {"case": "U001 SG duration unknown", "mode": document.mode, "primary": document.primary.plan_code},
        {
            "case": "U001 after March trip ended",
            "mode": known.mode,
            "primary": None if known.primary is None else known.primary.plan_code,
        },
        {"case": "U003 no destination", "mode": ask.mode, "primary": None},
        {"case": "U001 US no roam catalogue", "mode": none.mode, "primary": None},
    ]
)
invented = pd.DataFrame(
    [{"rejected_plan": "FAKE_PLAN", "in_ranked": "FAKE_PLAN" in [item.plan_code for item in document.ranked]}]
)
ranked.to_json(ROOT / "outputs" / "tables" / "u001_ranked.json", orient="records", indent=2)
modes.to_json(ROOT / "outputs" / "tables" / "decision_modes.json", orient="records", indent=2)
invented.to_json(ROOT / "outputs" / "tables" / "invented_plan_rejected.json", orient="records", indent=2)
metrics = {
    "mode": document.mode,
    "primary": document.primary.plan_code,
    "ranked": [item.plan_code for item in document.ranked],
    "uncertainty": [{"name": item.name, "status": item.status} for item in document.uncertainty],
}
(ROOT / "outputs" / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
modes


,case,mode,primary
0,U001 SG duration unknown,SCENARIO_BASED,ROAM_15
1,U001 after March trip ended,SCENARIO_BASED,ROAM_15
2,U003 no destination,ASK_FOR_INFORMATION,None
3,U001 US no roam catalogue,NO_RECOMMENDATION,None


In [4]:
fig, axis = plt.subplots(figsize=(6, 3.4))
axis.bar(ranked["plan_code"], ranked["score"], color=["#389e0d", "#d48806", "#1890ff"])
axis.set_title("U001 Singapore catalogue scores")
axis.set_ylabel("Score")
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "candidate_scores.png", dpi=120)
plt.close(fig)

status_counts = pd.Series([item.status for item in document.uncertainty]).value_counts()
fig, axis = plt.subplots(figsize=(5, 3.2))
axis.bar(status_counts.index, status_counts.values, color="#389e0d")
axis.set_title("Uncertainty statuses for U001 Singapore")
fig.tight_layout()
fig.savefig(ROOT / "outputs" / "plots" / "uncertainty_status.png", dpi=120)
plt.close(fig)
"plots written"


'plots written'